# Práctica guiada · Auditar el contrato

**Consigna:** auditar el contrato de la API: tipos, campos obligatorios, rango, versión, códigos de error y ejemplo. Proponer **tres pruebas** y **dos métricas operativas**.

Hay un servicio que predice el abandono de clientes de NovaRetail. Su contrato está en `contrato-api.yaml` y **tiene problemas**. Tu trabajo es encontrarlos probando pedidos, no leyendo el modelo.

**Pausas:** a los **8 minutos**, un control: mostrá un pedido que el contrato acepta y no debería. A los **15**, intercambiá el trabajo con otro equipo y pedí una observación concreta.

## Paso 0 · Instalación

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "yaml": "pyyaml", "joblib": "joblib", "matplotlib": "matplotlib"}
print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")

## Preparación

Esta celda carga las librerías, genera NovaRetail, entrena las versiones 1 y 2 del modelo y define el servicio. **No hace falta leerla**: correla una sola vez.

In [ ]:
# ═══════════════════════════ KIT DE PRODUCCIÓN · NovaRetail (datos ficticios, generados por código)
# De un notebook a un servicio: contrato, validación, versionado, registro de predicciones y monitoreo.
import hashlib, json, os, time, warnings
import numpy as np, pandas as pd, yaml, joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings("ignore")

def generar_novaretail(n=8000, semilla=2025, duplicados=60):
    rng = np.random.default_rng(semilla)
    fecha_corte = pd.Timestamp("2025-12-31")
    canal = rng.choice(["web", "app", "tienda"], size=n, p=[0.40, 0.25, 0.35])
    # composición: web concentra más clientes del segmento Esencial (esto confunde la lectura por canal)
    p_seg = {"web": [0.62, 0.28, 0.10], "app": [0.30, 0.45, 0.25], "tienda": [0.28, 0.42, 0.30]}
    segmento = np.array([rng.choice(["Esencial", "Plus", "Premium"], p=p_seg[c]) for c in canal])
    region = rng.choice(["Centro", "Norte", "Sur", "Litoral", "Cuyo"], size=n, p=[0.30, 0.20, 0.18, 0.17, 0.15])
    edad = np.clip(rng.normal(41, 12, n), 18, 80).round().astype(int)
    alta = pd.Timestamp("2021-01-01") + pd.to_timedelta(rng.integers(0, 1730, n), unit="D")
    antig = ((fecha_corte - alta).days / 30.4).round().astype(int)
    base_compras = np.select([segmento == "Esencial", segmento == "Plus"], [5, 9], 14)
    compras = rng.poisson(base_compras)
    ticket = np.round(np.select([segmento == "Esencial", segmento == "Plus"], [35, 62], 110) * rng.lognormal(0, 0.25, n), 2)
    dias = np.clip(rng.gamma(2.2, 24, n), 1, 365).round().astype(int)
    soporte = rng.poisson(np.where(segmento == "Esencial", 1.2, 0.6))
    usa_app = np.where(canal == "app", 1, (rng.random(n) < 0.35).astype(int))
    descuento = np.clip(rng.normal(np.where(segmento == "Esencial", 0.14, 0.08), 0.05, n), 0, 0.35).round(3)
    sat_real = np.clip(np.round(rng.normal(3.7 - 0.12 * soporte + 0.15 * (segmento == "Premium"), 0.9, n)), 1, 5)

    z_dias = (dias - 60) / 40
    logit = (-2.45
             + 0.85 * (segmento == "Esencial") - 0.45 * (segmento == "Premium")
             + 0.80 * z_dias + 0.30 * soporte - 0.42 * (sat_real - 3.7)
             - 0.50 * usa_app - 0.012 * (antig - 24) - 0.04 * (compras - 8)
             + 0.55 * ((canal == "tienda") & (segmento == "Esencial")))
    p = 1 / (1 + np.exp(-logit))
    abandono = (rng.random(n) < p).astype(int)

    # faltantes: la satisfacción falta más entre quienes abandonan (faltante informativo)
    sat = sat_real.copy()
    sat[rng.random(n) < np.where(abandono == 1, 0.30, 0.09)] = np.nan
    ticket = ticket.astype(float)
    ticket[rng.random(n) < 0.03] = np.nan
    region_s = pd.Series(region, dtype=object)
    region_s[rng.random(n) < 0.02] = np.nan
    # valores imposibles
    edad = edad.astype(float)
    malos = rng.random(n) < 0.004
    edad[malos] = rng.choice([0, 150, -1], size=malos.sum())
    # fuga: el motivo de baja solo existe para quienes ya abandonaron
    motivo = np.where(abandono == 1, rng.choice(["precio", "servicio", "competencia", "mudanza"], size=n), None)

    df = pd.DataFrame({
        "id_cliente": [f"NR{i:06d}" for i in range(1, n + 1)],
        "fecha_alta": alta.strftime("%Y-%m-%d"),
        "canal": canal, "segmento": segmento, "region": region_s,
        "edad": edad, "antiguedad_meses": antig, "compras_12m": compras,
        "ticket_promedio": ticket, "dias_desde_ultima_compra": dias,
        "contactos_soporte_90d": soporte, "usa_app": usa_app,
        "descuento_promedio": descuento, "satisfaccion": sat,
        "motivo_baja": motivo, "abandono_90d": abandono,
    })
    # duplicados: filas repetidas por un error de carga
    if duplicados:
        df = pd.concat([df, df.sample(duplicados, random_state=semilla)], ignore_index=True)
    return df.sample(frac=1, random_state=semilla).reset_index(drop=True)


def datos_limpios():
    df = generar_novaretail().drop_duplicates("id_cliente").reset_index(drop=True)
    df.loc[(df.edad < 18) | (df.edad > 100), "edad"] = np.nan
    df["satisfaccion_faltante"] = df.satisfaccion.isna().astype(int); return df
DF = datos_limpios()

NUM_SERV = ["edad", "antiguedad_meses", "compras_12m", "ticket_promedio", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app", "descuento_promedio", "satisfaccion"]
CAT_SERV = ["canal", "segmento", "region"]
NUM_MODELO = NUM_SERV + ["satisfaccion_faltante"]; OBJETIVO = "abandono_90d"

def armar_pipeline(modelo, num=NUM_MODELO, cat=CAT_SERV):
    pre = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num),
                             ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat)])
    return Pipeline([("pre", pre), ("modelo", modelo)])
def partir(df, columnas):
    return train_test_split(df[columnas], df[OBJETIVO], test_size=0.25, stratify=df[OBJETIVO], random_state=42)
_XTR, _XTE, _YTR, _YTE = partir(DF, NUM_MODELO + CAT_SERV)
MODELO_V1 = armar_pipeline(LogisticRegression(max_iter=3000)).fit(_XTR, _YTR)
MODELO_V2 = armar_pipeline(HistGradientBoostingClassifier(max_depth=3, random_state=7)).fit(_XTR, _YTR)
AUC_V1 = float(roc_auc_score(_YTE, MODELO_V1.predict_proba(_XTE)[:, 1])); AUC_V2 = float(roc_auc_score(_YTE, MODELO_V2.predict_proba(_XTE)[:, 1]))

# ═══════ EL CONTRATO: un archivo OpenAPI que describe qué entra y qué sale
CONTRATO_BORRADOR = """openapi: 3.0.3
info:
  title: Servicio de riesgo de abandono NovaRetail
  version: "0.1"
paths:
  /predict:
    post:
      summary: Probabilidad de abandono en 90 días
      requestBody:
        content:
          application/json:
            schema:
              $ref: "#/components/schemas/PrediccionRequest"
      responses:
        "200":
          description: Predicción
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/PrediccionResponse"
        "500":
          description: Error
components:
  schemas:
    PrediccionRequest:
      type: object
      required: [id_cliente]
      properties:
        id_cliente: {type: string}
        edad: {type: number}
        antiguedad_meses: {type: integer}
        compras_12m: {type: integer}
        ticket_promedio: {type: number}
        dias_desde_ultima_compra: {type: integer}
        contactos_soporte_90d: {type: integer}
        usa_app: {type: integer}
        descuento_promedio: {type: number}
        satisfaccion: {type: number}
        canal: {type: string}
        segmento: {type: string}
        region: {type: string}
    PrediccionResponse:
      type: object
      properties:
        probabilidad: {type: number}
"""
def leer_contrato(texto_o_ruta):
    texto = open(texto_o_ruta, encoding="utf-8").read() if os.path.exists(str(texto_o_ruta)) else texto_o_ruta
    return yaml.safe_load(texto)
def reglas_desde_contrato(contrato):
    esquema = contrato["components"]["schemas"]["PrediccionRequest"]
    return {"requeridos": list(esquema.get("required", [])), "propiedades": esquema.get("properties", {}), "extras_permitidos": esquema.get("additionalProperties", True) is not False}
def validar(payload, reglas):
    """Devuelve la lista de errores {campo, motivo}. Vacía = el payload cumple el contrato."""
    err = []; props = reglas["propiedades"]
    for c in reglas["requeridos"]:
        if c not in payload: err.append({"campo": c, "motivo": "falta"})
    for c, v in payload.items():
        if c not in props:
            if not reglas["extras_permitidos"]: err.append({"campo": c, "motivo": "campo no permitido"})
            continue
        p = props[c]
        if v is None:
            if not p.get("nullable"): err.append({"campo": c, "motivo": "no admite nulo"})
            continue
        t = p.get("type")
        if t == "integer" and (isinstance(v, bool) or not isinstance(v, int)): err.append({"campo": c, "motivo": "tiene que ser entero"}); continue
        if t == "number" and (isinstance(v, bool) or not isinstance(v, (int, float))): err.append({"campo": c, "motivo": "tiene que ser número"}); continue
        if t == "string" and not isinstance(v, str): err.append({"campo": c, "motivo": "tiene que ser texto"}); continue
        if "minimum" in p and v < p["minimum"]: err.append({"campo": c, "motivo": f"menor que {p['minimum']}"})
        if "maximum" in p and v > p["maximum"]: err.append({"campo": c, "motivo": f"mayor que {p['maximum']}"})
        if "enum" in p and v not in p["enum"]: err.append({"campo": c, "motivo": f"valor no permitido (admite {p['enum']})"})
    return err
# ═══════ EL SERVICIO: carga del artefacto, esquema, transformación, inferencia, respuesta y registro
def transformar(payload):
    fila = {c: pd.to_numeric(payload.get(c), errors="coerce") for c in NUM_SERV}
    fila["satisfaccion_faltante"] = int(pd.isna(fila["satisfaccion"])); fila.update({c: (payload.get(c) if payload.get(c) is not None else np.nan) for c in CAT_SERV})
    return pd.DataFrame([fila])
def hash_entrada(payload): return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:10]
class Servicio:
    """Simula un POST /v1/predict: valida contra el contrato, puntúa con una versión del modelo y deja un registro."""
    def __init__(self, contrato, modelo=None, version="v1"):
        self.contrato = contrato if isinstance(contrato, dict) else leer_contrato(contrato); self.reglas = reglas_desde_contrato(self.contrato)
        self.modelo, self.version, self.registro, self.relojes = modelo or MODELO_V1, version, [], 0
    def responder(self, payload):
        t0 = time.perf_counter(); errores = validar(payload, self.reglas)
        if errores: resp = {"status": 422, "errores": errores}
        else:
            try: resp = {"status": 200, "probabilidad": round(float(self.modelo.predict_proba(transformar(payload))[0, 1]), 4)}
            except Exception as e: resp = {"status": 500, "detalle": type(e).__name__}
        resp["version_modelo"] = self.version; resp["version_contrato"] = str(self.contrato["info"]["version"])
        self.relojes += 1; self.registro.append({"n": self.relojes, "version_modelo": self.version, "status": resp["status"], "entrada": hash_entrada(payload), "probabilidad": resp.get("probabilidad"), "latencia_ms": (time.perf_counter() - t0) * 1000})
        return resp
def cliente_ejemplo(i=0, **cambios):
    f = DF.iloc[i]; p = {c: (None if pd.isna(f[c]) else (int(f[c]) if c in ("antiguedad_meses", "compras_12m", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app") else float(f[c]))) for c in NUM_SERV}
    p.update({c: f[c] for c in CAT_SERV}); p["id_cliente"] = f["id_cliente"]; p.update(cambios); return p
def _tipico():
    p = cliente_ejemplo(0); p.update({"edad": 41.0, "antiguedad_meses": 30, "compras_12m": 7, "ticket_promedio": 48.5, "dias_desde_ultima_compra": 55, "contactos_soporte_90d": 1, "usa_app": 1, "descuento_promedio": 0.1, "satisfaccion": 4.0, "canal": "web", "segmento": "Plus", "region": "Centro"}); return p
CASOS_CONTRATO = [("típico, todo válido", _tipico(), False), ("satisfacción nula (permitida)", {**_tipico(), "satisfaccion": None}, False),
    ("falta un campo que usa el modelo", {k: v for k, v in _tipico().items() if k != "dias_desde_ultima_compra"}, True), ("texto en un campo numérico", {**_tipico(), "compras_12m": "muchas"}, True),
    ("edad imposible (150)", {**_tipico(), "edad": 150.0}, True), ("edad imposible (−1)", {**_tipico(), "edad": -1.0}, True), ("días negativos", {**_tipico(), "dias_desde_ultima_compra": -5}, True),
    ("segmento que no existe", {**_tipico(), "segmento": "Gold"}, True), ("canal nuevo sin avisar", {**_tipico(), "canal": "marketplace"}, True),
    ("campo con fuga (motivo_baja)", {**_tipico(), "motivo_baja": "precio"}, True), ("campo extra inesperado", {**_tipico(), "cupon": "X10"}, True),
    ("descuento fuera de rango", {**_tipico(), "descuento_promedio": 1.7}, True), ("usa_app fuera de {0, 1}", {**_tipico(), "usa_app": 2}, True), ("payload vacío", {}, True)]
def probar_reglas(reglas, mostrar=True):
    filas = []
    for nombre, payload, debe_rechazar in CASOS_CONTRATO:
        rechaza = bool(validar(payload, reglas)); ok = rechaza == debe_rechazar
        filas.append({"caso": nombre, "debería": "rechazar" if debe_rechazar else "aceptar", "el contrato": "rechaza" if rechaza else "acepta", "ok": "✔" if ok else "✘"})
    t = pd.DataFrame(filas)
    if mostrar: print(t.to_string(index=False))
    return t

# ═══════ EL REGISTRO DE VERSIONES: artefacto + ficha
def registrar_version(carpeta, version, modelo, auc, algoritmo, semilla=None):
    ruta = os.path.join(carpeta, version); os.makedirs(ruta, exist_ok=True); joblib.dump(modelo, os.path.join(ruta, "modelo.joblib"))
    ficha = {"version": version, "algoritmo": algoritmo, "auc_prueba": round(auc, 4), "columnas_entrada": NUM_MODELO + CAT_SERV, "version_contrato": "1.0.0", "semilla": semilla,
             "hash_datos_entrenamiento": hashlib.sha256(pd.util.hash_pandas_object(DF[NUM_MODELO + CAT_SERV + [OBJETIVO]], index=False).values.tobytes()).hexdigest()[:12], "filas_entrenamiento": int(len(_XTR))}
    json.dump(ficha, open(os.path.join(ruta, "ficha.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2); return ficha
def cargar_version(carpeta, version): return joblib.load(os.path.join(carpeta, version, "modelo.joblib")), json.load(open(os.path.join(carpeta, version, "ficha.json"), encoding="utf-8"))

def escribir_contratos(carpeta=".", revisado=False):
    """Deja el contrato borrador (y, si se pide, el revisado) como archivos .yaml en la carpeta."""
    os.makedirs(carpeta, exist_ok=True); open(os.path.join(carpeta, "contrato-api.yaml"), "w", encoding="utf-8").write(CONTRATO_BORRADOR)
    if revisado: open(os.path.join(carpeta, "contrato-api.revisado.yaml"), "w", encoding="utf-8").write(CONTRATO_REVISADO)

# ═══════ CASOS CONOCIDOS PARA LA PRUEBA DE HUMO
def _golden(clase):
    base = _tipico(); tip = [{**base, "dias_desde_ultima_compra": d, "segmento": s, "contactos_soporte_90d": c} for d, s, c in ((40, "Plus", 1), (50, "Esencial", 2), (55, "Premium", 0), (45, "Plus", 3), (60, "Esencial", 1))]
    bor = [{**base, "dias_desde_ultima_compra": d, "segmento": s, "contactos_soporte_90d": c} for d, s, c in ((90, "Esencial", 3), (120, "Plus", 2), (150, "Esencial", 4), (180, "Premium", 1), (240, "Esencial", 5), (300, "Plus", 3))]
    return {"tipicos": tip, "borde": bor, "mezcla": tip[:3] + bor[:3]}[clase]
def _probas(modelo, casos, defecto=False):
    return np.array([float(modelo.predict_proba(transformar({**c, "dias_desde_ultima_compra": min(c["dias_desde_ultima_compra"], 60)} if defecto else c))[0, 1]) for c in casos])


if not os.path.exists("contrato-api.yaml"): escribir_contratos(".")      # deja el contrato borrador junto al notebook si no estaba
pd.options.display.width = 200; print(f"Listo: {len(DF):,} clientes · versión 1 del modelo con AUC {AUC_V1:.3f}".replace(",", "."))

## 1 · Leé el contrato

Está escrito en YAML. Fijate en `required`, en `properties` y en `responses`.

In [ ]:
print(open("contrato-api.yaml", encoding="utf-8").read())

## 2 · Probá pedidos

`probar_pedido` manda un pedido al servicio con **este** contrato y muestra qué responde. Partí del pedido típico y cambiá una cosa por vez.

In [ ]:
servicio = Servicio("contrato-api.yaml")
def probar_pedido(**cambios):
    p = {**_tipico(), **cambios}; quitar = [k for k, v in cambios.items() if v == "__quitar__"]
    for k in quitar: p.pop(k, None)
    r = servicio.responder(p); print({k: v for k, v in r.items() if k in ("status", "probabilidad", "errores")}); return r
probar_pedido()                                   # el pedido típico
# probar_pedido(edad=150.0)                       # 👉 probá otras variantes: un campo que falta ("__quitar__"), uno de más, un valor imposible…
# probar_pedido(region="__quitar__")

## 3 · Tus hallazgos

Escribí, en una frase cada uno, **qué le falta al contrato**. Una pista para ordenarte: tipos, campos obligatorios, rangos, valores permitidos, campos extra, versión, códigos de error y ejemplo.

Después proponé **tres pruebas** (un pedido, qué esperás y por qué) y **dos métricas operativas** para el servicio en producción.

In [ ]:
# 👉 TU TURNO
hallazgos = [
    # "El contrato acepta ... porque ...",
]
pruebas = [
    # "Pedido sin dias_desde_ultima_compra: espero 422 porque el modelo lo necesita",
]
metricas = [
    # "...",
]

## 4 · La revisión

`revisar_auditoria` **no corrige**: devuelve preguntas. Respondelas y volvé a correrla.

In [ ]:
import re

def revisar_auditoria(hallazgos, pruebas, metricas):
    cat = {"campos obligatorios": r"obligatori|requerid|required|falta", "rangos": r"rango|m[ií]nimo|m[aá]ximo|imposible|negativ|150", "valores permitidos": r"enum|permitid|valores|categor|segmento|canal|gold|marketplace",
           "campos extra": r"extra|desconoc|additional|no permitid|sobra|fuga|motivo_baja", "versión": r"versi[oó]n|/v1|v1", "códigos de error": r"422|c[oó]digo|error|400|rechaz", "ejemplo": r"ejemplo|example"}
    pistas = []; cubiertas = {k for k, rx in cat.items() if any(re.search(rx, h, re.I) for h in hallazgos)}
    if len(hallazgos) < 4: pistas.append("Tenés menos de cuatro hallazgos: ¿qué más le podés probar al contrato?")
    for k in cat:
        if k not in cubiertas: pistas.append(f"Ningún hallazgo habla de «{k}»: ¿qué pasa con el contrato en ese punto?")
    if len(pruebas) < 3: pistas.append("Faltan pruebas: ¿cuáles son las tres que más te protegerían?")
    elif sum(1 for t in pruebas if re.search(r"espero|esper|debe|422|200|rechaz|acept", t, re.I)) < 3: pistas.append("Cada prueba tiene que decir qué esperás que pase: ¿qué respuesta esperás en cada una?")
    ops = {"latencia": r"latencia|p95|tiempo", "rechazos": r"rechaz|422|error", "volumen": r"volumen|pedidos|tr[aá]fico", "drift": r"drift|psi|cambio de datos", "versión": r"versi[oó]n"}
    if len(metricas) < 2: pistas.append("Faltan métricas operativas: ¿qué mirarías todos los días?")
    elif sum(1 for m in metricas if any(re.search(rx, m, re.I) for rx in ops.values())) < 2: pistas.append("Tus métricas no son operativas: ¿cuáles te dicen si el servicio funciona, no si el modelo es bueno?")
    print(f"Revisión: {len(pistas)} pregunta(s) pendiente(s).")
    for q in pistas: print("  👉", q)
    if not pistas: print("✔ Sin preguntas pendientes.")
    return pistas
# pistas = revisar_auditoria(hallazgos, pruebas, metricas)